In [1]:
import pandas as pd
import os

PASTA_DADOS = "."

ENCODING = "latin-1"
SEP = ";"

pd.set_option("display.max_columns", None)

In [2]:
for arquivo in os.listdir(PASTA_DADOS):
    print(arquivo)

perfil_eleitorado_2018
transferencia_temporaria_2018.zip
baixar_tse.sh
eleitorado_local_votacao_2018.zip
perfil_comparecimento_abstencao_2018.zip
perfil_comparecimento_abstencao_eleitor_deficiente_2018.zip
perfil_comparecimento_abstencao_2018
perfil_comparecimento_abstencao_eleitor_tte_2018
transf_temporaria_secao_2018
.venv
perfil_eleitorado_2018.zip
perfil_eleitor_deficiencia_2018
perfil_comparecimento_abstencao_eleitor_deficiente_2018
perfil_comparecimento_abstencao_eleitor_tte_2018.zip
transferencia_temporaria_2018
eleitorado_local_votacao_2018
pipeline_tse_2018.ipynb
perfil_eleitor_deficiencia_2018.zip
transf_temporaria_secao_2018.zip


In [3]:
def carregar_csv(caminho, nrows=None):
    return pd.read_csv(caminho, sep=SEP, encoding=ENCODING, nrows=nrows, low_memory=False)

df_comparecimento = carregar_csv(
    os.path.join(PASTA_DADOS, "perfil_comparecimento_abstencao_2018", "perfil_comparecimento_abstencao_2018_BRASIL.csv")
)

df_eleitorado = carregar_csv(
    os.path.join(PASTA_DADOS, "perfil_eleitorado_2018", "perfil_eleitorado_2018.csv")
)

In [4]:
print("Colunas - Comparecimento/Abstenção:")
print(df_comparecimento.columns.tolist())
print()
print("Colunas - Eleitorado (perfil geral):")
print(df_eleitorado.columns.tolist())

Colunas - Comparecimento/Abstenção:
['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'NR_TURNO', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'CD_GENERO', 'DS_GENERO', 'CD_ESTADO_CIVIL', 'DS_ESTADO_CIVIL', 'CD_FAIXA_ETARIA', 'DS_FAIXA_ETARIA', 'CD_GRAU_ESCOLARIDADE', 'DS_GRAU_ESCOLARIDADE', 'CD_COR_RACA', 'DS_COR_RACA', 'CD_QUILOMBOLA', 'DS_QUILOMBOLA', 'CD_INTERPRETE_LIBRAS', 'DS_INTERPRETE_LIBRAS', 'CD_IDENTIDADE_GENERO', 'DS_IDENTIDADE_GENERO', 'CD_IDIOMA_INDIGENA', 'DS_IDIOMA_INDIGENA', 'CD_GRUPO_INDIGENA', 'DS_GRUPO_INDIGENA', 'QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCAO', 'QT_COMPARECIMENTO_DEFICIENCIA', 'QT_ABSTENCAO_DEFICIENCIA', 'QT_COMPARECIMENTO_TTE', 'QT_ABSTENCAO_TTE', 'QT_COMPAREC_FACULTATIVO', 'QT_ABST_FACULTATIVO', 'QT_COMPAREC_OBRIGATORIO', 'QT_ABST_OBRIGATORIO', 'QT_COMPAREC_DEFIC_FACULTATIVO', 'QT_ABST_DEFIC_FACULTATIVO', 'QT_COMPAREC_DEFIC_OBRIGATORIO', 'QT_ABST_DEFIC_OBRIGATORIO']

Colunas - Eleitorado (perfil geral):
['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO',

In [5]:
def tem_granularidade_secao(df):
    colunas_secao = [c for c in df.columns if "SECAO" in c.upper()]
    return len(colunas_secao) > 0, colunas_secao

tem_secao_comp, cols_comp = tem_granularidade_secao(df_comparecimento)
tem_secao_elei, cols_elei = tem_granularidade_secao(df_eleitorado)

print(f"Comparecimento tem seção eleitoral? {tem_secao_comp} -> {cols_comp}")
print(f"Eleitorado (geral) tem seção eleitoral? {tem_secao_elei} -> {cols_elei}")

if tem_secao_comp and not tem_secao_elei:
    print("\n>>> ATENÇÃO: você provavelmente PRECISA baixar os arquivos por UF")
    print(">>> ('Perfil do eleitorado por seção eleitoral') para casar a granularidade.")
elif tem_secao_comp and tem_secao_elei:
    print("\n>>> As duas bases têm seção eleitoral — pode fazer o join direto, sem precisar dos arquivos por UF.")
else:
    print("\n>>> Nenhuma das duas tem seção eleitoral explícita — o join deve ser feito por município/zona.")

Comparecimento tem seção eleitoral? False -> []
Eleitorado (geral) tem seção eleitoral? False -> []

>>> Nenhuma das duas tem seção eleitoral explícita — o join deve ser feito por município/zona.


In [6]:
def padronizar_colunas(df):
    df = df.copy()
    df.columns = [c.strip().upper() for c in df.columns]
    return df

df_comparecimento = padronizar_colunas(df_comparecimento)
df_eleitorado = padronizar_colunas(df_eleitorado)

In [7]:
CHAVE_JOIN = [
    "SG_UF",
    "CD_MUNICIPIO",
    "NR_ZONA",
    "CD_GENERO",
    "CD_ESTADO_CIVIL",
    "CD_FAIXA_ETARIA",
    "CD_GRAU_ESCOLARIDADE",
]

In [ ]:
# Cubo de Dados -- OBS: Observar novamente os nulos. 
# Criar tabela stagin e Temp no projeto

df_final = df_eleitorado.merge(
    df_comparecimento,
    on=CHAVE_JOIN,
    how="left",
    suffixes=("_eleitorado", "_comparecimento"),
)

print(f"Linhas eleitorado: {len(df_eleitorado)}")
print(f"Linhas comparecimento: {len(df_comparecimento)}")
print(f"Linhas após merge: {len(df_final)}")
print(f"Linhas com NaN em QT_COMPARECIMENTO: {df_final['QT_COMPARECIMENTO'].isna().sum()}")
df_final.head()

Linhas eleitorado: 4181293
Linhas comparecimento: 8377682
Linhas após merge: 8377682
Linhas com NaN em QT_COMPARECIMENTO: 0


,DT_GERACAO_eleitorado,HH_GERACAO_eleitorado,ANO_ELEICAO_eleitorado,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO_eleitorado,CD_MUN_SIT_BIOMETRICA,DS_MUN_SIT_BIOMETRICA,NR_ZONA,CD_GENERO,DS_GENERO_eleitorado,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL_eleitorado,CD_FAIXA_ETARIA,DS_FAIXA_ETARIA_eleitorado,CD_GRAU_ESCOLARIDADE,DS_GRAU_ESCOLARIDADE_eleitorado,QT_ELEITORES_PERFIL,QT_ELEITORES_BIOMETRIA,QT_ELEITORES_DEFICIENCIA,QT_ELEITORES_INC_NM_SOCIAL,DT_GERACAO_comparecimento,HH_GERACAO_comparecimento,ANO_ELEICAO_comparecimento,NR_TURNO,NM_MUNICIPIO_comparecimento,DS_GENERO_comparecimento,DS_ESTADO_CIVIL_comparecimento,DS_FAIXA_ETARIA_comparecimento,DS_GRAU_ESCOLARIDADE_comparecimento,CD_COR_RACA,DS_COR_RACA,CD_QUILOMBOLA,DS_QUILOMBOLA,CD_INTERPRETE_LIBRAS,DS_INTERPRETE_LIBRAS,CD_IDENTIDADE_GENERO,DS_IDENTIDADE_GENERO,CD_IDIOMA_INDIGENA,DS_IDIOMA_INDIGENA,CD_GRUPO_INDIGENA,DS_GRUPO_INDIGENA,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCAO,QT_COMPARECIMENTO_DEFICIENCIA,QT_ABSTENCAO_DEFICIENCIA,QT_COMPARECIMENTO_TTE,QT_ABSTENCAO_TTE,QT_COMPAREC_FACULTATIVO,QT_ABST_FACULTATIVO,QT_COMPAREC_OBRIGATORIO,QT_ABST_OBRIGATORIO,QT_COMPAREC_DEFIC_FACULTATIVO,QT_ABST_DEFIC_FACULTATIVO,QT_COMPAREC_DEFIC_OBRIGATORIO,QT_ABST_DEFIC_OBRIGATORIO
0,12/04/2021,13:55:01,2018,RJ,58653,NITERÓI,1,Biométrico,71,4,FEMININO,1,SOLTEIRO,5054,50 a 54 anos,7,SUPERIOR INCOMPLETO,126,125,1,0,30/04/2025,07:09:09,2018,1,NITERÓI,FEMININO,SOLTEIRO,50 a 54 anos,SUPERIOR INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,126,108,18,1,0,0,0,0,0,108,18,0,0,1,0
1,12/04/2021,13:55:01,2018,RJ,58653,NITERÓI,1,Biométrico,71,4,FEMININO,1,SOLTEIRO,5054,50 a 54 anos,7,SUPERIOR INCOMPLETO,126,125,1,0,30/04/2025,07:09:09,2018,2,NITERÓI,FEMININO,SOLTEIRO,50 a 54 anos,SUPERIOR INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,126,107,19,1,0,0,0,0,0,107,19,0,0,1,0
2,12/04/2021,13:55:01,2018,RJ,58653,NITERÓI,1,Biométrico,71,4,FEMININO,1,SOLTEIRO,5559,55 a 59 anos,7,SUPERIOR INCOMPLETO,101,101,2,0,30/04/2025,07:09:09,2018,1,NITERÓI,FEMININO,SOLTEIRO,55 a 59 anos,SUPERIOR INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,101,94,7,2,0,0,0,0,0,94,7,0,0,2,0
3,12/04/2021,13:55:01,2018,RJ,58653,NITERÓI,1,Biométrico,71,4,FEMININO,1,SOLTEIRO,5559,55 a 59 anos,7,SUPERIOR INCOMPLETO,101,101,2,0,30/04/2025,07:09:09,2018,2,NITERÓI,FEMININO,SOLTEIRO,55 a 59 anos,SUPERIOR INCOMPLETO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,101,90,11,2,0,0,0,0,0,90,11,0,0,2,0
4,12/04/2021,13:55:01,2018,RJ,58653,NITERÓI,1,Biométrico,71,4,FEMININO,1,SOLTEIRO,6569,65 a 69 anos,2,LÊ E ESCREVE,57,57,4,0,30/04/2025,07:09:09,2018,1,NITERÓI,FEMININO,SOLTEIRO,65 a 69 anos,LÊ E ESCREVE,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,57,43,14,3,1,0,0,0,0,43,14,0,0,3,1


In [ ]:
dados_tratados = df_comparecimento.groupby()[]